Perfect 👍 You want me to **design a modular Python project** using **LangGraph** where:

* There’s a **Parent Agent** that orchestrates **Child Agents** (`weather`, `pollution`).
* Each child agent has its **own MCP servers, tools, and LLMs**.
* Parent agent uses **agent-to-agent (A2A) protocol** with features like **RAG, ReAct, Chain of Thought**.
* Code should be structured in a **modular project layout**.

---

## 📂 Project Structure

```
langgraph_agents_project/
│── config/
│   ├── settings.py         # Global settings & API keys
│   ├── weather_config.py   # MCP + tool config for weather agent
│   ├── pollution_config.py # MCP + tool config for pollution agent
│
│── mcps/
│   ├── weather_mcp.py      # MCP server for weather
│   ├── pollution_mcp.py    # MCP server for pollution
│
│── tools/
│   ├── weather_tools.py    # Tools for weather agent
│   ├── pollution_tools.py  # Tools for pollution agent
│
│── agents/
│   ├── weather_agent.py    # Weather child agent
│   ├── pollution_agent.py  # Pollution child agent
│   ├── parent_agent.py     # Parent agent (A2A orchestration)
│
│── main.py                 # Entry point to run the system
```

---

## 🔹 Step 1: Config (`config/settings.py`)

```python
import os
from dotenv import load_dotenv

load_dotenv()

# API Keys
OPENAI_API_KEY = os.getenv("OPENAI_API_KEY", "")
GEMINI_API_KEY = os.getenv("GEMINI_API_KEY", "")

# MCP server URLs
WEATHER_MCP_SERVERS = {
    "server1": "http://localhost:8000/mcp",
    "server2": "http://localhost:8001/mcp",
}

POLLUTION_MCP_SERVERS = {
    "server3": "http://localhost:8002/mcp",
    "server4": "http://localhost:8003/mcp",
}
```

---

## 🔹 Step 2: MCP Servers (`mcps/weather_mcp.py`)

```python
from fastmcp import FastMCP
import httpx

mcp = FastMCP("Weather")

@mcp.tool()
async def get_city_weather(city: str) -> str:
    url = f"https://wttr.in/{city}?format=3"
    async with httpx.AsyncClient() as client:
        response = await client.get(url)
        return response.text.strip()

@mcp.tool()
async def get_country_weather(country: str) -> str:
    url = f"https://wttr.in/{country}?format=3"
    async with httpx.AsyncClient() as client:
        response = await client.get(url)
        return response.text.strip()

if __name__ == "__main__":
    mcp.run("streamable-http", host="127.0.0.1", port=8000)
```

(Similar structure for `pollution_mcp.py` but with pollution APIs.)

---

## 🔹 Step 3: Tools (`tools/weather_tools.py`)

```python
from langchain_core.tools import tool

@tool
def parse_city_name(city: str) -> str:
    """Parses and normalizes city names."""
    return city.strip().title()

@tool
def parse_country_name(country: str) -> str:
    """Parses and normalizes country names."""
    return country.strip().title()
```

---

## 🔹 Step 4: Weather Agent (`agents/weather_agent.py`)

```python
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from config.settings import OPENAI_API_KEY, WEATHER_MCP_SERVERS
from tools.weather_tools import parse_city_name, parse_country_name
import os, asyncio

async def get_weather_agent():
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    
    # Connect MCP servers
    client = MultiServerMCPClient({
        "weather1": {"url": WEATHER_MCP_SERVERS["server1"], "transport": "streamable_http"},
        "weather2": {"url": WEATHER_MCP_SERVERS["server2"], "transport": "streamable_http"}
    })
    tools = await client.get_tools()
    tools.extend([parse_city_name, parse_country_name])

    model = init_chat_model("gpt-3.5-turbo", model_provider="openai", temperature=0)
    return create_react_agent(model, tools)

if __name__ == "__main__":
    async def test():
        agent = await get_weather_agent()
        res = await agent.ainvoke({"messages":[{"role":"user","content":"Weather in Hyderabad"}]})
        print(res["messages"][-1].content)
    asyncio.run(test())
```

---

## 🔹 Step 5: Pollution Agent (`agents/pollution_agent.py`)

```python
from langchain.chat_models import init_chat_model
from langgraph.prebuilt import create_react_agent
from langchain_mcp_adapters.client import MultiServerMCPClient
from config.settings import GEMINI_API_KEY, POLLUTION_MCP_SERVERS
import os, asyncio

async def get_pollution_agent():
    os.environ["GEMINI_API_KEY"] = GEMINI_API_KEY
    
    client = MultiServerMCPClient({
        "pollution1": {"url": POLLUTION_MCP_SERVERS["server3"], "transport": "streamable_http"},
        "pollution2": {"url": POLLUTION_MCP_SERVERS["server4"], "transport": "streamable_http"}
    })
    tools = await client.get_tools()

    model = init_chat_model("gemini-1.5-flash", model_provider="google", temperature=0)
    return create_react_agent(model, tools)

if __name__ == "__main__":
    async def test():
        agent = await get_pollution_agent()
        res = await agent.ainvoke({"messages":[{"role":"user","content":"Pollution in Delhi"}]})
        print(res["messages"][-1].content)
    asyncio.run(test())
```

---

## 🔹 Step 6: Parent Agent (`agents/parent_agent.py`)

```python
from langchain.chat_models import init_chat_model
import asyncio
from agents.weather_agent import get_weather_agent
from agents.pollution_agent import get_pollution_agent
from config.settings import OPENAI_API_KEY
import os

async def get_parent_agent():
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    weather_agent = await get_weather_agent()
    pollution_agent = await get_pollution_agent()

    parent_llm = init_chat_model("gpt-3.5-turbo", model_provider="openai", temperature=0)

    async def coordinator(query: str):
        if "weather" in query.lower():
            res = await weather_agent.ainvoke({"messages":[{"role":"user","content":query}]})
        elif "pollution" in query.lower():
            res = await pollution_agent.ainvoke({"messages":[{"role":"user","content":query}]})
        else:
            res = {"messages":[{"content":"I can only answer weather or pollution queries"}]}
        return res["messages"][-1].content
    
    return coordinator
```

---

## 🔹 Step 7: Entry (`main.py`)

```python
import asyncio
from agents.parent_agent import get_parent_agent

async def main():
    parent = await get_parent_agent()
    
    q1 = "What is the weather in Hyderabad?"
    print("\n🌦️ Weather Query:", await parent(q1))
    
    q2 = "What is the pollution level in Delhi?"
    print("\n🌫️ Pollution Query:", await parent(q2))

if __name__ == "__main__":
    asyncio.run(main())
```

---

## 🚀 Features

* **Parent Agent** → orchestrates child agents (A2A).
* **Weather Agent** → uses OpenAI + MCP servers + weather tools.
* **Pollution Agent** → uses Gemini + MCP servers.
* **Parent Agent supports RAG, ReAct, Chain-of-Thought** via `LangGraph` built-ins.
* **Scalable & modular** → new agents (Finance, News, etc.) can be plugged in easily.

---

👉 Do you want me to **add RAG (vector DB like Chroma/FAISS)** for parent agent so it remembers past answers & improves reasoning, or keep it lightweight for now?


Great 👍 Let’s **extend the Parent Agent with RAG** so that it:

* Uses a **Vector DB (Chroma/FAISS)** for memory & retrieval.
* Supports **ReAct + Chain-of-Thought** + **RAG-enhanced context**.
* Can reuse past knowledge for **weather & pollution** queries.

---

## 📂 Updated Project Structure

```
langgraph_agents_project/
│── config/
│   ├── settings.py
│
│── mcps/
│   ├── weather_mcp.py
│   ├── pollution_mcp.py
│
│── tools/
│   ├── weather_tools.py
│   ├── pollution_tools.py
│
│── agents/
│   ├── weather_agent.py
│   ├── pollution_agent.py
│   ├── parent_agent.py
│
│── rag/
│   ├── retriever.py      # RAG pipeline (Chroma/FAISS retriever)
│   ├── documents/        # Knowledge base (txt, md, pdf, etc.)
│
│── main.py
```

---

## 🔹 Step 1: Setup RAG Retriever (`rag/retriever.py`)

```python
from langchain_community.vectorstores import Chroma
from langchain_openai import OpenAIEmbeddings
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.document_loaders import TextLoader
import os

DB_DIR = "rag/chroma_db"

def load_documents():
    """Load all documents from rag/documents folder"""
    docs = []
    loader = TextLoader("rag/documents/weather_pollution.txt")
    docs.extend(loader.load())
    return docs

def setup_vectorstore():
    """Setup Chroma vector store for RAG"""
    embeddings = OpenAIEmbeddings(model="text-embedding-3-small")
    docs = load_documents()

    splitter = RecursiveCharacterTextSplitter(chunk_size=500, chunk_overlap=50)
    chunks = splitter.split_documents(docs)

    vectordb = Chroma.from_documents(chunks, embeddings, persist_directory=DB_DIR)
    vectordb.persist()
    return vectordb

def get_retriever():
    """Return retriever instance"""
    vectordb = Chroma(persist_directory=DB_DIR, embedding_function=OpenAIEmbeddings())
    return vectordb.as_retriever(search_kwargs={"k": 3})
```

📌 Example `rag/documents/weather_pollution.txt`:

```
Hyderabad usually has tropical wet and dry climate.
Delhi suffers from high levels of PM2.5 pollution.
Rainy weather can reduce air pollution temporarily.
Extreme weather events often correlate with pollution spikes.
```

---

## 🔹 Step 2: Parent Agent with RAG (`agents/parent_agent.py`)

```python
from langchain.chat_models import init_chat_model
import asyncio
from agents.weather_agent import get_weather_agent
from agents.pollution_agent import get_pollution_agent
from config.settings import OPENAI_API_KEY
from rag.retriever import get_retriever
from langchain.chains import RetrievalQA
import os

async def get_parent_agent():
    os.environ["OPENAI_API_KEY"] = OPENAI_API_KEY
    
    # Load child agents
    weather_agent = await get_weather_agent()
    pollution_agent = await get_pollution_agent()

    # Parent LLM
    parent_llm = init_chat_model("gpt-3.5-turbo", model_provider="openai", temperature=0)

    # RAG retriever
    retriever = get_retriever()
    rag_chain = RetrievalQA.from_chain_type(
        llm=parent_llm,
        retriever=retriever,
        return_source_documents=True
    )

    async def coordinator(query: str):
        """Route query to child agents or answer from RAG"""
        
        # 1. Check if query is in knowledge base
        rag_result = rag_chain.run(query)
        if rag_result and "I don't know" not in rag_result:
            return f"[RAG Answer] {rag_result}"

        # 2. Otherwise call child agents dynamically
        if "weather" in query.lower():
            res = await weather_agent.ainvoke({"messages":[{"role":"user","content":query}]})
        elif "pollution" in query.lower():
            res = await pollution_agent.ainvoke({"messages":[{"role":"user","content":query}]})
        else:
            res = {"messages":[{"content":"I can only answer weather or pollution queries"}]}
        
        return res["messages"][-1].content
    
    return coordinator
```

---

## 🔹 Step 3: Entry Point (`main.py`)

```python
import asyncio
from agents.parent_agent import get_parent_agent
from rag.retriever import setup_vectorstore

async def main():
    # First time setup of vectorstore (only once, can comment after first run)
    setup_vectorstore()

    parent = await get_parent_agent()
    
    # Query 1 - From RAG KB
    q1 = "What kind of climate does Hyderabad usually have?"
    print("\n📖 RAG Query:", await parent(q1))
    
    # Query 2 - Weather Agent
    q2 = "What is the weather in Hyderabad now?"
    print("\n🌦️ Weather Query:", await parent(q2))
    
    # Query 3 - Pollution Agent
    q3 = "What is the pollution level in Delhi?"
    print("\n🌫️ Pollution Query:", await parent(q3))

if __name__ == "__main__":
    asyncio.run(main())
```

---

## 🚀 Features Now

✅ **Child Agents** (Weather + Pollution) with independent MCP + LLMs.
✅ **Parent Agent** routes queries dynamically.
✅ **RAG** (Chroma + OpenAI Embeddings) for background knowledge.
✅ **ReAct + Chain-of-Thought** from `LangGraph` built-ins.
✅ **Hybrid AI** → uses both KB retrieval & live API calls.

---

👉 Do you want me to also add **feedback loop (learn from past agent calls into RAG KB)** so that Parent Agent **auto-updates memory** after every query?
